> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 8 · Notebook 07 — The Deflated Sharpe Ratio and the probability of backtest overfitting

**Sessions:** S13 (The Deflated Sharpe Ratio) · S14 (Probability of backtest overfitting, CSCV) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Compute how high the best Sharpe of N worthless strategies is expected to be.
2. Deflate a Sharpe ratio for the number of trials behind it.
3. Measure the probability that the in-sample winner is a loser out of sample (PBO).

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()

## 1. The best of 200 worthless strategies

Two hundred strategies of pure noise: every true Sharpe is exactly zero. Six years of daily data each. Pick the best.

In [ ]:
noise = p.noise_strategies(n_days=1500, n=200)
srs = np.array([p.sharpe(noise[:, i]) for i in range(noise.shape[1])])
best = int(np.argmax(srs))
print(f"best annual Sharpe {srs[best]:.2f} (true Sharpe: 0); {np.mean(srs > 0.5):.0%} of the 200 look better than 0.5")
plt.hist(srs, bins=30); plt.axvline(srs[best], color=p.PALETTE[7]); plt.title("Sharpe ratios of 200 noise strategies"); plt.show()

How high should the best of `N` unskilled trials be? With `V` the variance of the trial Sharpes (per period, ddof=1) and `γ ≈ 0.5772` the Euler–Mascheroni constant (`p.EULER`):

`E[max SR] ≈ √V · ((1 − γ)·Φ⁻¹(1 − 1/N) + γ·Φ⁻¹(1 − 1/(N·e)))`

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from scipy.stats import norm

def expected_max_sharpe(trial_srs):
    s = np.asarray(trial_srs, dtype=float)
    n, v = s.size, s.var(ddof=1)
    return float(np.sqrt(v) * ((1 - p.EULER) * norm.ppf(1 - 1 / n) + p.EULER * norm.ppf(1 - 1 / (n * np.e))))

per_period = srs / np.sqrt(252)
cases = [per_period, per_period[:20], np.r_[per_period, -per_period]]     # 200, the first 20, and 400 (with mirror images)
mine = [p.attempt(expected_max_sharpe, s) for s in cases]
mine = p.check("expected_max_sharpe", mine, [p.expected_max_sharpe(s) for s in cases])
print(f"expected best annual Sharpe of pure noise: N=200 → {mine[0] * np.sqrt(252):.2f}; N=20 → {mine[1] * np.sqrt(252):.2f}; N=400 → {mine[2] * np.sqrt(252):.2f}")

## 2. Deflating the winner

The **Deflated Sharpe Ratio** is the PSR of the chosen strategy (notebook 04) with the benchmark raised from 0 to that expected maximum `SR0`. It answers: *given how many things I tried, how likely is this Sharpe to be real?* Return `(DSR, SR0)`; the usual bar is DSR >= 0.95.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def deflated_sharpe(returns, trial_srs):
    sr0 = p.expected_max_sharpe(trial_srs)
    return p.psr(returns, sr0), sr0

skilled = p.noise_strategies(n_days=1500, n=200, seed=7, skilled=1, skill_sharpe=1.5)[:, 0]
cases = [(noise[:, best], per_period), (skilled, per_period), (noise[:, best], per_period[:5])]
mine = [p.attempt(deflated_sharpe, *cs) for cs in cases]
mine = p.check("deflated_sharpe", mine, [p.deflated_sharpe(*cs) for cs in cases])
pd.DataFrame({"strategy": ["best of 200 noise", "a real Sharpe-1.5 strategy", "best of 200 noise, but claiming 5 trials"],
              "annual Sharpe": [p.sharpe(x) for x, _ in cases], "PSR vs 0": [p.psr(x) for x, _ in cases],
              "DSR": [d for d, _ in mine]}).round(3)

The ordinary PSR says the noise winner is very likely real. The DSR, which knows about the 199 siblings, says it isn't. Under-report the trials (the third row) and the DSR is fooled too: that's why the research log (notebook 01) records every run.

The second row is the uncomfortable one: a strategy with a **true** Sharpe of 1.5 happened to realize only 0.75 over these six years, and as one of 200 tries it can't be told apart from luck either. Deflation is strict by design; the answer is more evidence (longer history, other markets, paper trading), not fewer recorded trials.

## 3. The probability of backtest overfitting (CSCV)

Split the history into `S` blocks. For every way of choosing half the blocks as in-sample: pick the configuration with the best in-sample Sharpe, and find its **relative rank** out of sample, `w = (rank among the N OOS Sharpes, 1 = worst) / (N + 1)`, and its logit `ln(w / (1 − w))`. PBO is the share of logits `<= 0`: how often the in-sample winner lands at or below the OOS median. Write the rank-and-logit step.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import itertools

In [ ]:
def pbo_cscv(perf, S=10):
    T, N = perf.shape
    blocks = np.array_split(np.arange(T), S)
    sr = lambda x: x.mean(0) / x.std(0, ddof=1)
    logits = []
    for is_blocks in itertools.combinations(range(S), S // 2):
        is_idx = np.concatenate([blocks[i] for i in is_blocks])
        oos_idx = np.concatenate([blocks[i] for i in range(S) if i not in is_blocks])
        best = int(np.argmax(sr(perf[is_idx])))
        rank = sr(perf[oos_idx]).argsort().argsort()[best] + 1        # 1 = worst out of sample
        logits.append(np.log(rank / (N + 1) / (1 - rank / (N + 1))))
    logits = np.array(logits)
    return float(np.mean(logits <= 0)), logits

configs = noise[:, :50]
mine = p.attempt(pbo_cscv, configs)
mine = p.check("pbo_cscv", mine, p.pbo_cscv(configs, S=10))
print(f"PBO of 50 noise configurations: {mine[0]:.2f}")

One sample of noise gives a noisy PBO. Repeat it over several independent noise sets, and compare with sets where five of the 50 configurations have a real edge:

In [ ]:
rows = []
for seed in range(8):
    rows.append({"seed": seed, "50 noise configs": p.pbo_cscv(p.noise_strategies(1500, 50, seed=seed), S=10)[0],
                 "5 skilled + 45 noise": p.pbo_cscv(p.noise_strategies(1500, 50, seed=seed, skilled=5, skill_sharpe=1.5), S=10)[0]})
t = pd.DataFrame(rows).set_index("seed")
display(t.round(2).T)
print(f"mean PBO: noise {t.iloc[:, 0].mean():.2f}, with real skill {t.iloc[:, 1].mean():.2f}")

Noise averages around 0.4–0.5 (the in-sample winner is roughly a coin flip out of sample), and any single estimate can land far from it. Real skill pushes PBO toward 0. Use PBO alongside DSR, never alone.

## Wrap-up

* The more you try, the higher the best result by luck: deflate for the number of trials, and log them all.
* PBO asks whether in-sample ranking survives out of sample.
* Graded version: `labs/part08/week28_overfitting` and Clinic W4 (the validation dossier).